# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [43]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Charger les données

In [44]:
import json
import ast
from pathlib import Path

def load_pyomo_data(input_path="./data/pyomo_data.json"):
    """Charge les données JSON et convertit les clés string en types natifs."""
    input_file = Path(input_path)
    with open(input_file, "r") as f:
        data = json.load(f)

    def _convert_key(key):
        if not isinstance(key, str):
            return key
        if key.startswith("(") and key.endswith(")"):
            try:
                return ast.literal_eval(key)
            except Exception:
                return key
        try:
            return int(key)
        except Exception:
            return key

    # Convertir les dictionnaires de parametres indexes
    params = data.get("params", {})
    for pname, pval in list(params.items()):
        if isinstance(pval, dict):
            params[pname] = {_convert_key(k): v for k, v in pval.items()}

    cartesian = data.get("cartesian_data", {})
    for cname, cval in list(cartesian.items()):
        if isinstance(cval, dict):
            cartesian[cname] = {_convert_key(k): v for k, v in cval.items()}

    data["params"] = params
    data["cartesian_data"] = cartesian
    return data

# Charger les données
data = load_pyomo_data()

## 🔹 Model

In [45]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [46]:
model.MACHINES = Set(initialize=data['sets']['MACHINES'])
model.PRODUITS = Set(initialize=data['sets']['PRODUITS'])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.MACHINES for j in model.PRODUITS])

## 🔹 Parameters

In [47]:
model.disponibilite = Param(model.MACHINES, initialize=data['params']['disponibilite'], within=NonNegativeReals)
model.gain = Param(model.PRODUITS, initialize=data['params']['gain'], within=NonNegativeReals)
model.temps = Param(model.MACHINES, model.PRODUITS, initialize=data['cartesian_data']['temps'], within=NonNegativeReals)

## 🔹 Variables

In [48]:
model.x = Var(model.PRODUITS, domain=NonNegativeReals)

## 🔹 Constraints

In [49]:
model.c_for_0 = ConstraintList()
for m in model.MACHINES:
    model.c_for_0.add(sum(model.temps[m,p] * model.x[p] for p in model.PRODUITS) <= model.disponibilite[m])

## 🔹 Objective

In [50]:
model.obj = Objective(expr=sum(model.gain[p] * model.x[p] for p in model.PRODUITS), sense=maximize)

## ⚙️ Résolution du modèle

In [51]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

✅ Solver status: ok
✅ Termination condition: optimal


## 🎯 Valeur de la fonction objective

In [52]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

Objectif: obj
Valeur optimale: 825.0000
Sens: Maximisation


## 📊 Valeurs optimales des variables

In [53]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')

,Variable,Index,Valeur
0,x,remorcage,2.5000
1,x,stabilisateur,3.3333
